In [1]:
import pandas as pd
import pandas_ta as ta
import requests 
from backtesting import Strategy
from backtesting import Backtest
import json
from flask import Flask, jsonify, request
from backtesting.lib import crossover

C:\Users\fatma\anaconda3\lib\site-packages\backtesting\_plotting.py:50: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support (e.g. PyCharm, Spyder IDE). Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

In [2]:
key ='7D4D2IVSL1NS5WR3'

In [3]:
url = 'https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol=IBM&outputsize=full&apikey={key}&datatype=csv'

In [4]:
df = pd.read_csv(url)
df = pd.DataFrame(df)
df.shape

(6330, 6)

In [5]:
df['RSI'] = ta.rsi(df.close, length=14)

In [6]:
df['EMA_slow'] = ta.ema(df.close, length=200)#sma slow moving average
df['EMA_fast'] = ta.ema(df.close, length=150)#sma slow moving average

In [7]:
my_bbands = ta.bbands(df.close, length=14, std=2.0)
my_bbands[0:50]
df=df.join(my_bbands)
df.dropna(inplace=True)
df.isna().sum()

timestamp     0
open          0
high          0
low           0
close         0
volume        0
RSI           0
EMA_slow      0
EMA_fast      0
BBL_14_2.0    0
BBM_14_2.0    0
BBU_14_2.0    0
BBB_14_2.0    0
BBP_14_2.0    0
dtype: int64

In [8]:
df.reset_index(inplace=True)

In [9]:
def addemasignal(df):
    emasignal = [0]*len(df)
    for i in range(0, len(df)):
        if df.EMA_slow[i]>df.EMA_fast[i]:
            emasignal[i]=2
        elif df.EMA_slow[i]<df.EMA_fast[i]:
            emasignal[i]=1
    df['EMASignal'] = emasignal
addemasignal(df)  

In [10]:
def addorderslimit(df, percent):
    ordersignal=[0]*len(df)
    for i in range(1, len(df)): #EMASignal of previous candle 
        if df.close[i]<=df['BBL_14_2.0'][i] and df.EMASignal[i]==2:
            ordersignal[i]=df.close[i]-df.close[i]*percent
        elif df.close[i]>=df['BBU_14_2.0'][i] and df.EMASignal[i]==1:
            ordersignal[i]=df.close[i]+df.close[i]*percent
    df['ordersignal']=ordersignal
addorderslimit(df,0.000) 

In [11]:
dfcopy = df[:].copy()
def SIGNAL():
     return dfcopy.ordersignal

In [12]:
dfcopy = dfcopy.rename(columns = {'open': 'Open', 'high': 'High', 'low': 'Low', 'close': 'Close', 'volume': 'Volume'})


In [13]:
class MyStrat (Strategy):
    initsize = 0.99
    mysize = initsize
    def init(self):
        super().init()
        self.signal = self.I(SIGNAL)
    def next(self):
        super().next()
        TPSLRatio = 2
        perc = 0.02

        if len(self.trades)>0:
            if self.data.index[-1]-self.trades[-1].entry_time>=10:
                self.trades[-1].close()
            if self.trades[-1].is_long and self.data.RSI[-1]>=75:
                self.trades[-1].close()
            elif self.trades[-1].is_short and self.data.RSI[-1]<=25:
                self.trades[-1].close()
                
                
        if self.signal!=0 and len(self.trades)==0 and self.data.EMASignal==2:
            sl1 = min(self.data.Low[-1],self.data.Low[-2])*(1-perc)
            tp1 = self.data.Close[-1]+(self.data.Close[-1] - sl1) *TPSLRatio
            self.buy(sl=sl1, tp=tp1, size=self.mysize)
            
        elif self.signal!=0 and len(self.trades)== 0 and self.data.EMASignal==1:
            sl1 = sl1 = max(self.data.High[-1],self.data.High[-2])*(1+perc)
            tp1 = self.data.Close[-1]-(sl1 - self.data.Close[-1])*TPSLRatio
            self.sell(sl=sl1, tp=tp1, size=self.mysize)


In [14]:
bt = Backtest(dfcopy, MyStrat, cash=1000, margin=1/5 , commission=.000)
stat = bt.run()
stat

C:\Users\fatma\AppData\Local\Temp\ipykernel_68248\1003485627.py:1: UserWarning: Data index is not datetime. Assuming simple periods, but `pd.DateTimeIndex` is advised.
  bt = Backtest(dfcopy, MyStrat, cash=1000, margin=1/5 , commission=.000)


Start                                     0.0
End                                    6130.0
Duration                               6130.0
Exposure Time [%]                    24.18855
Equity Final [$]                    82.724872
Equity Peak [$]                      7623.445
Return [%]                         -91.727513
Buy & Hold Return [%]              -49.981906
Return (Ann.) [%]                         0.0
Volatility (Ann.) [%]                     NaN
Sharpe Ratio                              NaN
Sortino Ratio                             NaN
Calmar Ratio                              0.0
Max. Drawdown [%]                  -99.373972
Avg. Drawdown [%]                  -22.989063
Max. Drawdown Duration                 2931.0
Avg. Drawdown Duration             254.086957
# Trades                                208.0
Win Rate [%]                        43.269231
Best Trade [%]                      17.893333
Worst Trade [%]                    -11.854839
Avg. Trade [%]                    

In [15]:
bt.plot()

Row(id='1455', ...)

In [16]:
def add_ma_crossover_signal(df):
        df['SMA_50'] = ta.sma(df['close'], window=50)  # 50-day Simple Moving Average
        df['SMA_200'] = ta.sma(df['close'], window=200)  # 200-day Simple Moving Average
        df['MA_Crossover'] = ta.utils.cross(df['SMA_50'], df['SMA_200'])


In [17]:
def signal_ma_crossover(df):
    add_ma_crossover_signal(df)
    return df['MA_Crossover'] 

In [18]:
add_ma_crossover_signal(df)
dfcopy = df[:].copy()

In [19]:
def SIGNAL():
    return dfcopy.SMA_200

In [20]:
dfcopy = dfcopy.rename(columns = {'open': 'Open', 'high': 'High', 'low': 'Low', 'close': 'Close', 'volume': 'Volume'})


In [21]:
class MovingAverageStrategy(Strategy):
        initsize = 0.99
        mysize = initsize

        def init(self):
            super().init()
            self.signal = self.I(SIGNAL)

        def next(self):
            super().next()
            TPSLRatio = 2
            perc = 0.02

            if len(self.trades) > 0:
            # For example, if the price goes against the position or based on a specific indicator
                 pass

            if self.signal > 0 and len(self.trades) == 0:
            # Buy signal - Place buy trade
                sl1 = self.data.Close[-1] * (1 - perc)
                tp1 = self.data.Close[-1] + (self.data.Close[-1] - sl1) * TPSLRatio
                self.buy(sl=sl1, tp=tp1, size=self.mysize)

            elif self.signal < 0 and len(self.trades) == 0:
            # Sell signal - Place sell trade
                sl1 = self.data.Close[-1] * (1 + perc)
                tp1 = self.data.Close[-1] - (sl1 - self.data.Close[-1]) * TPSLRatio
                self.sell(sl=sl1, tp=tp1, size=self.mysize)

In [22]:
dfcopy.index = pd.to_datetime(dfcopy.index)

In [23]:
bt = Backtest(dfcopy, MovingAverageStrategy, cash=5000, margin=1/5 , commission=.000)
stat = bt.run()
stat

Start                     1970-01-01 00:00:00
End                       1970-01-01 00:00...
Duration                  0 days 00:00:00....
Exposure Time [%]                   28.135704
Equity Final [$]                      10.2786
Equity Peak [$]                     6675.8012
Return [%]                         -99.794428
Buy & Hold Return [%]              -49.981906
Return (Ann.) [%]                         0.0
Volatility (Ann.) [%]                     NaN
Sharpe Ratio                              NaN
Sortino Ratio                             NaN
Calmar Ratio                              0.0
Max. Drawdown [%]                  -99.846032
Avg. Drawdown [%]                  -40.622604
Max. Drawdown Duration    0 days 00:00:00....
Avg. Drawdown Duration    0 days 00:00:00....
# Trades                                  441
Win Rate [%]                        24.943311
Best Trade [%]                       8.691925
Worst Trade [%]                     -8.965046
Avg. Trade [%]                    

In [24]:
bt.plot()


C:\Users\fatma\AppData\Local\Temp\ipykernel_68248\3833524480.py:1: UserWarning: 'Can't superimpose OHLC data with rule 'None'(index datetime resolution: 'microsecond'). Skipping.
  bt.plot()


Row(id='2194', ...)

In [25]:
# Renommer les colonnes pour standardiser
dfcopy.rename(columns={'open': 'Open','high': 'High','low': 'Low','close': 'Close','volume': 'Volume'}, inplace=True)

# Vérifier les premières lignes
print(dfcopy.head())

                               index   timestamp    Open     High     Low  \
1970-01-01 00:00:00.000000000    199  2024-03-14  196.95  197.748  192.12   
1970-01-01 00:00:00.000000001    200  2024-03-13  197.55  198.100  195.32   
1970-01-01 00:00:00.000000002    201  2024-03-12  192.46  199.180  192.15   
1970-01-01 00:00:00.000000003    202  2024-03-11  195.09  195.380  190.88   
1970-01-01 00:00:00.000000004    203  2024-03-08  196.06  197.770  194.38   

                                Close   Volume        RSI    EMA_slow  \
1970-01-01 00:00:00.000000000  193.43  4102202  65.686025  197.069750   
1970-01-01 00:00:00.000000001  196.70  3960737  70.605177  197.066071   
1970-01-01 00:00:00.000000002  197.78  5862512  72.031285  197.073175   
1970-01-01 00:00:00.000000003  191.73  4712688  55.722301  197.020009   
1970-01-01 00:00:00.000000004  195.95  3943113  62.158328  197.009362   

                                 EMA_fast  BBL_14_2.0  BBM_14_2.0  BBU_14_2.0  \
1970-01-01 00:00:

In [26]:
# Fonction Ichimoku personnalisée
def add_ichimoku(df):
    df['tenkan_sen'] = (df['High'].rolling(window=9).max() + df['Low'].rolling(window=9).min()) / 2
    df['kijun_sen'] = (df['High'].rolling(window=26).max() + df['Low'].rolling(window=26).min()) / 2
    df['senkou_span_a'] = ((df['tenkan_sen'] + df['kijun_sen']) / 2).shift(26)
    df['senkou_span_b'] = ((df['High'].rolling(window=52).max() + df['Low'].rolling(window=52).min()) / 2).shift(26)
    df['chikou_span'] = df['Close'].shift(-26)
    df.dropna(inplace=True)

add_ichimoku(dfcopy)

# Afficher les premières lignes pour vérifier
print(dfcopy[['tenkan_sen', 'kijun_sen', 'senkou_span_a', 'senkou_span_b', 'chikou_span']].head())


                               tenkan_sen  kijun_sen  senkou_span_a  \
1970-01-01 00:00:00.000000077    158.1500    159.925      169.21250   
1970-01-01 00:00:00.000000078    156.4700    159.345      169.21250   
1970-01-01 00:00:00.000000079    155.4750    159.345      169.21250   
1970-01-01 00:00:00.000000080    154.8200    159.235      169.21250   
1970-01-01 00:00:00.000000081    153.2675    158.370      169.40875   

                               senkou_span_b  chikou_span  
1970-01-01 00:00:00.000000077       178.5325       139.21  
1970-01-01 00:00:00.000000078       178.5325       138.46  
1970-01-01 00:00:00.000000079       178.5325       141.24  
1970-01-01 00:00:00.000000080       178.3075       143.23  
1970-01-01 00:00:00.000000081       178.3075       142.11  


In [27]:
class IchimokuStrategy(Strategy):
    def init(self):
        price = self.data.Close
        self.tenkan_sen = self.I(lambda: self.data.tenkan_sen)
        self.kijun_sen = self.I(lambda: self.data.kijun_sen)
        self.senkou_span_a = self.I(lambda: self.data.senkou_span_a)
        self.senkou_span_b = self.I(lambda: self.data.senkou_span_b)
        self.chikou_span = self.I(lambda: self.data.chikou_span)

    def next(self):
        # Signal d'achat
        if (self.tenkan_sen[-1] > self.kijun_sen[-1] and
            self.data.Close[-1] > self.senkou_span_a[-1] and
            self.data.Close[-1] > self.senkou_span_b[-1] and
            self.chikou_span[-1] > self.data.Close[-1]):
            self.buy()

        # Signal de vente
        elif (self.tenkan_sen[-1] < self.kijun_sen[-1] and
                self.data.Close[-1] < self.senkou_span_a[-1] and
                self.data.Close[-1] < self.senkou_span_b[-1] and
                self.chikou_span[-1] < self.data.Close[-1]):
            self.sell()

In [28]:
bt = Backtest(dfcopy, IchimokuStrategy, cash=10000, margin=1/5, commission=0.0005)
stat = bt.run()

# Afficher les statistiques
print(stat)

Start                     1970-01-01 00:00...
End                       1970-01-01 00:00...
Duration                  0 days 00:00:00....
Exposure Time [%]                    3.898474
Equity Final [$]                          0.0
Equity Peak [$]                  30704.961535
Return [%]                             -100.0
Buy & Hold Return [%]              -23.149893
Return (Ann.) [%]                         0.0
Volatility (Ann.) [%]                     NaN
Sharpe Ratio                              NaN
Sortino Ratio                             NaN
Calmar Ratio                              0.0
Max. Drawdown [%]                      -100.0
Avg. Drawdown [%]                   -15.70811
Max. Drawdown Duration    0 days 00:00:00....
Avg. Drawdown Duration    0 days 00:00:00....
# Trades                                   14
Win Rate [%]                         7.142857
Best Trade [%]                       1.725277
Worst Trade [%]                    -17.857395
Avg. Trade [%]                    

In [29]:
bt.plot()

C:\Users\fatma\AppData\Local\Temp\ipykernel_68248\651457420.py:1: UserWarning: 'Can't superimpose OHLC data with rule 'None'(index datetime resolution: 'microsecond'). Skipping.
  bt.plot()


Row(id='2949', ...)

In [40]:
def add_macd(df):
    macd = ta.macd(df['close'], fast=12, slow=26, signal=9)
    df['MACD'] = macd['MACD_12_26_9']
    df['MACD_Signal'] = macd['MACDs_12_26_9']
    df['MACD_Hist'] = macd['MACDh_12_26_9']
    df.dropna(inplace=True)


In [41]:
class MACDStrategy(Strategy):
    initsize = 0.99
    mysize = initsize

    def init(self):
        super().init()
        self.macd = self.I(lambda: self.data.MACD)
        self.signal = self.I(lambda: self.data.MACD_Signal)

    def next(self):
        super().next()
        
        if len(self.trades) > 0:
            # Fermer la position si le MACD croise la ligne de signal à l'inverse
            if self.trades[-1].is_long and self.macd[-1] < self.signal[-1]:
                self.trades[-1].close()
            elif self.trades[-1].is_short and self.macd[-1] > self.signal[-1]:
                self.trades[-1].close()

        # Achat lorsque MACD croise la ligne Signal à la hausse
        if self.macd[-1] > self.signal[-1] and len(self.trades) == 0:
            self.buy(size=self.mysize)

        # Vente lorsque MACD croise la ligne Signal à la baisse
        elif self.macd[-1] < self.signal[-1] and len(self.trades) == 0:
            self.sell(size=self.mysize)

In [38]:
print(dfcopy.columns)

Index(['index', 'timestamp', 'Open', 'High', 'Low', 'Close', 'Volume', 'RSI',
       'EMA_slow', 'EMA_fast', 'BBL_14_2.0', 'BBM_14_2.0', 'BBU_14_2.0',
       'BBB_14_2.0', 'BBP_14_2.0', 'EMASignal', 'ordersignal', 'SMA_50',
       'SMA_200', 'MA_Crossover', 'tenkan_sen', 'kijun_sen', 'senkou_span_a',
       'senkou_span_b', 'chikou_span'],
      dtype='object')


In [39]:
dfcopy.columns = dfcopy.columns.str.strip().str.lower()
print(dfcopy.columns)

Index(['index', 'timestamp', 'open', 'high', 'low', 'close', 'volume', 'rsi',
       'ema_slow', 'ema_fast', 'bbl_14_2.0', 'bbm_14_2.0', 'bbu_14_2.0',
       'bbb_14_2.0', 'bbp_14_2.0', 'emasignal', 'ordersignal', 'sma_50',
       'sma_200', 'ma_crossover', 'tenkan_sen', 'kijun_sen', 'senkou_span_a',
       'senkou_span_b', 'chikou_span'],
      dtype='object')


In [42]:
add_macd(dfcopy)  # Appliquer le calcul du MACD

# Renommer les colonnes pour Backtesting
dfcopy = dfcopy.rename(columns={'open': 'Open', 'high': 'High', 'low': 'Low', 'close': 'Close', 'volume': 'Volume'})


In [43]:
bt = Backtest(dfcopy, MACDStrategy, cash=1000, margin=1/5, commission=0.0005)
stat = bt.run()

# Afficher les statistiques
print(stat)

Start                     1970-01-01 00:00...
End                       1970-01-01 00:00...
Duration                  0 days 00:00:00....
Exposure Time [%]                   54.495413
Equity Final [$]                     10.52886
Equity Peak [$]                   1683.869835
Return [%]                         -98.947114
Buy & Hold Return [%]              -16.421707
Return (Ann.) [%]                         0.0
Volatility (Ann.) [%]                     NaN
Sharpe Ratio                              NaN
Sortino Ratio                             NaN
Calmar Ratio                              0.0
Max. Drawdown [%]                  -99.374722
Avg. Drawdown [%]                  -45.018731
Max. Drawdown Duration    0 days 00:00:00....
Avg. Drawdown Duration    0 days 00:00:00....
# Trades                                  222
Win Rate [%]                        40.540541
Best Trade [%]                      22.061108
Worst Trade [%]                    -11.877261
Avg. Trade [%]                    

In [44]:
bt.plot()

C:\Users\fatma\AppData\Local\Temp\ipykernel_68248\651457420.py:1: UserWarning: 'Can't superimpose OHLC data with rule 'None'(index datetime resolution: 'microsecond'). Skipping.
  bt.plot()


Row(id='3745', ...)